In [1]:
import numpy as np 
import pandas as pd 
import ipaddress # For preprocesing ip adresses
import re # regular expression library will be used to extract values within brackets
from sklearn.preprocessing import MinMaxScaler # used to normalize continuous data in physical dataset

# I / Preprocessing 

### 0- Concat and Merge compatibility

In [7]:
df1 = pd.read_csv('Network_dataset/csv/attack_1.csv')

In [9]:
df2 = pd.read_csv('Network_dataset/csv/attack_2.csv')

In [10]:
df3 = pd.read_csv('Network_dataset/csv/attack_3.csv')

In [11]:
df4 = pd.read_csv('Network_dataset/csv/attack_4.csv')

In [17]:
df_phy1 = pd.read_csv('Physical_dataset/phy_att_1.csv', encoding='utf-16', delimiter='\t')

In [18]:
df_phy2 = pd.read_csv('Physical_dataset/phy_att_2.csv', encoding='utf-16', delimiter='\t')

In [52]:
df_phy3 = pd.read_csv('Physical_dataset/phy_att_3.csv', encoding='utf-16', delimiter='\t')

In [66]:
df_phy4 = pd.read_csv('Physical_dataset/phy_att_4.csv')

In [11]:
df = pd.read_csv('super.csv')

Check the Time part where we will rearange them

### 1- Network Data

In [12]:
# Display features of out dataset
print(df.columns)

Index(['mac_s', 'mac_d', 'ip_s', 'ip_d', 'sport', 'dport', 'proto', 'flags',
       'size', 'modbus_fn', 'n_pkt_src', 'n_pkt_dst', 'modbus_response',
       'Tank_1', 'Tank_2', 'Tank_3', 'Tank_4', 'Tank_5', 'Tank_6', 'Tank_7',
       'Tank_8', 'Pump_1', 'Pump_2', 'Pump_3', 'Pump_4', 'Pump_5', 'Pump_6',
       'Flow_sensor_1', 'Flow_sensor_2', 'Flow_sensor_3', 'Flow_sensor_4',
       'Valv_1', 'Valv_2', 'Valv_3', 'Valv_4', 'Valv_5', 'Valv_6', 'Valv_7',
       'Valv_8', 'Valv_9', 'Valv_10', 'Valv_11', 'Valv_12', 'Valv_13',
       'Valv_14', 'Valv_15', 'Valv_16', 'Valv_17', 'Valv_18', 'Valv_19',
       'Valv_20', 'Valv_21', 'Valv_22', 'Label'],
      dtype='object')


In [30]:
# Display all nan values per feature
print(df.isna().sum())

mac_s                    0
mac_d                    0
ip_s                   945
ip_d                   945
sport                22775
dport                22775
proto                    0
flags                22775
size                     0
modbus_fn           961018
n_pkt_src              945
n_pkt_dst              945
modbus_response    2348433
Tank_1                   0
Tank_2                   0
Tank_3                   0
Tank_4                   0
Tank_5                   0
Tank_6                   0
Tank_7                   0
Tank_8                   0
Pump_1                   0
Pump_2                   0
Pump_3                   0
Pump_4                   0
Pump_5                   0
Pump_6                   0
Flow_sensor_1            0
Flow_sensor_2            0
Flow_sensor_3            0
Flow_sensor_4            0
Valv_1                   0
Valv_2                   0
Valv_3                   0
Valv_4                   0
Valv_5                   0
Valv_6                   0
V

In [31]:
# Iterate over each feature to print the number of unique values and the unique values themselves
for column in df.columns:
    unique_count = df[column].nunique(dropna=False)  # Set dropna=False to include NaN in the count if present
    unique_values = df[column].unique()
    
    print(f"Feature: {column}")
    print(f"Number of Unique Values: {unique_count}")
    print(f"Unique Values: {unique_values}\n")

Feature: mac_s
Number of Unique Values: 9
Unique Values: ['74:46:a0:bd:a7:1b' '0a:fe:ec:47:74:fb' 'fa:00:bc:90:d7:fa'
 'e6:3f:ac:c9:a8:8c' '00:80:f4:03:fb:12' 'fe:bb:16:7b:c3:27'
 '4a:35:83:e0:3d:a4' '00:0c:29:47:8c:22' '00:0c:29:47:8c:0e']

Feature: mac_d
Number of Unique Values: 10
Unique Values: ['0a:fe:ec:47:74:fb' 'e6:3f:ac:c9:a8:8c' 'fa:00:bc:90:d7:fa'
 '74:46:a0:bd:a7:1b' '00:80:f4:03:fb:12' 'fe:bb:16:7b:c3:27'
 '4a:35:83:e0:3d:a4' 'ff:ff:ff:ff:ff:ff' '00:0c:29:47:8c:22'
 '00:0c:29:47:8c:0e']

Feature: ip_s
Number of Unique Values: 9
Unique Values: ['84.3.251.20' '84.3.251.102' '84.3.251.103' '84.3.251.101' '84.3.251.18'
 '84.3.251.105' '84.3.251.104' nan '84.3.251.110']

Feature: ip_d
Number of Unique Values: 9
Unique Values: ['84.3.251.102' '84.3.251.101' '84.3.251.103' '84.3.251.20' '84.3.251.18'
 '84.3.251.105' '84.3.251.104' nan '84.3.251.110']

Feature: sport
Number of Unique Values: 65537
Unique Values: [56667. 56666. 56668. ...  1129.  1130.  1131.]

Feature: dport
Numbe

In [13]:
# Check for any -1 values across all features because later will be used to replace nan values and to reinforce the missing_pattern
print((df == -1).any().any())

# As you can see there is no single -1 value which will allow us to proceed as planned

False


In [14]:
# Clean column names by removing spaces that will later create errors in calls
df.columns = df.columns.str.replace(' ', '')

In [15]:
# set function using the ipadress library to transform ips to original decimal representation
def ip_to_int(ip):
    if pd.isna(ip):
        return -1  # in order to keep the nan values intact we replace it by -1 as we will do the same with all other nans
    return int(ipaddress.ip_address(ip))


# Apply the function to the IP address columns
df['ip_s'] = df['ip_s'].apply(ip_to_int)
df['ip_d'] = df['ip_d'].apply(ip_to_int)


In [16]:
# Factorize excluding -1 to keep our nan values remarkable as -1
def factorize_exclude_neg1(series):
    # Identify the values to be factorized (excluding -1)
    mask = series != -1
    
    # Factorize the masked values
    factorized_values, unique_values = pd.factorize(series[mask])
    
    # Create a full series with -1 preserved
    result = pd.Series(-1, index=series.index)
    result[mask] = factorized_values
    return result

# Apply the factorization while preserving -1
df['ip_s'] = factorize_exclude_neg1(df['ip_s'])
df['ip_d'] = factorize_exclude_neg1(df['ip_d'])

In [38]:
# as you can see nan values disappeared in ip_s and ip_d
print(df.isna().sum())

mac_s                     0
mac_d                     0
ip_s                      0
ip_d                      0
sport               4115050
dport               4115050
proto                     0
flags               4115050
size                      0
modbus_fn           5078450
n_pkt_src              1373
n_pkt_dst              1373
modbus_response    13319822
Tank_1                    0
Tank_2                    0
Tank_3                    0
Tank_4                    0
Tank_5                    0
Tank_6                    0
Tank_7                    0
Tank_8                    0
Pump_1                    0
Pump_2                    0
Pump_3                    0
Pump_4                    0
Pump_5                    0
Pump_6                    0
Flow_sensor_1             0
Flow_sensor_2             0
Flow_sensor_3             0
Flow_sensor_4             0
Valv_1                    0
Valv_2                    0
Valv_3                    0
Valv_4                    0
Valv_5              

In [39]:
# and here we ve got existence of -1  True since they replaced nan values in previous columns ip
print((df == -1).any().any())

True


In [17]:
# Drop the 'dport' column and keep 'sport' because of negative correlation and specialist advice
# The 'dport' and 'sport' columns are negatively correlated and often alternate values, introducing redundancy.
# retaining 'sport' because it is more valuable for detecting certain attacks
# where 'sport' might not receive a corresponding 'dport' response, making 'sport' crucial for identifying anomalies.
df = df.drop(columns=['dport'])

In [18]:
# Replace all nan values in the 'sport' column with -1
df['sport'] = df['sport'].fillna(-1)

In [19]:
# Replace all nan values in the 'flags' column with -1
df['flags'] = df['flags'].fillna(-1)

In [20]:
# Same goes for 'n_pkt_src' & 'n_pkt_dst' 
df['n_pkt_src'] = df['n_pkt_src'].fillna(-1)
df['n_pkt_dst'] = df['n_pkt_dst'].fillna(-1)

In [21]:
# Function to extract integers from brackets and also replace nan with -1 in modbus_response
def extract_int(value):
    if pd.isna(value):
        return -1  # Replace NaNs with -1
    match = re.search(r'\d+', str(value))
    if match:
        return int(match.group(0))
    return -1

# Apply the function to the 'modbus_response' column
df['modbus_response'] = df['modbus_response'].apply(extract_int)

In [22]:
# Function to factorize the mac_s and mac_d address columns and get unique values id's 
def factorize_mac_column(col):
    factorized_values, unique_values = pd.factorize(col)
    return factorized_values

# Apply the function to the MAC address columns
df['mac_s'] = factorize_mac_column(df['mac_s'])
df['mac_d'] = factorize_mac_column(df['mac_d'])

In [46]:
# as you can see now the only one remaining with nan values is modbus_fn which we will proceed next
print(df.isna().sum())

mac_s                    0
mac_d                    0
ip_s                     0
ip_d                     0
sport                    0
proto                    0
flags                    0
size                     0
modbus_fn          5078450
n_pkt_src                0
n_pkt_dst                0
modbus_response          0
Tank_1                   0
Tank_2                   0
Tank_3                   0
Tank_4                   0
Tank_5                   0
Tank_6                   0
Tank_7                   0
Tank_8                   0
Pump_1                   0
Pump_2                   0
Pump_3                   0
Pump_4                   0
Pump_5                   0
Pump_6                   0
Flow_sensor_1            0
Flow_sensor_2            0
Flow_sensor_3            0
Flow_sensor_4            0
Valv_1                   0
Valv_2                   0
Valv_3                   0
Valv_4                   0
Valv_5                   0
Valv_6                   0
Valv_7                   0
V

In [23]:
# Replace NaNs with a placeholder value for factorization
modbus_fn_temp = df['modbus_fn'].fillna('NaN_Placeholder')

# Factorize the column
factorized_values, unique_values = pd.factorize(modbus_fn_temp)

# Replace the placeholder with -1
factorized_values = np.where(modbus_fn_temp == 'NaN_Placeholder', -1, factorized_values)

# Add the processed column back to the DataFrame
df['modbus_fn'] = factorized_values

In [24]:
# Factorize the proto column
proto_factorized, proto_unique = pd.factorize(df['proto'])

# Overwrite column like we did in all previous ones
df['proto'] = proto_factorized

In [25]:
# Factorize the size column
size_factorized, size_unique = pd.factorize(df['size'])

# Overwrite column like we did in all previous ones
df['size'] = size_factorized

In [26]:
# Factorize the flags column that contains binary representation of flags, we transform them to simple binary representation
def factorize_flags(series):
    # Identify the values to be factorized (excluding -1)
    mask = series != -1
    
    # Factorize the masked values
    factorized_values, unique_values = pd.factorize(series[mask])
    
    # Create a full series with -1 preserved
    result = pd.Series(-1, index=series.index)
    result[mask] = factorized_values
    return result
df['flags'] = factorize_flags(df['flags'])

In [ ]:
# Drop Time feature since the packets are already organized by time corresponding their index 
df = df.drop(['Time'], axis=1)
# DO NOT EXE ALREADY TIMELESS DF

In [28]:
df['sport'] = df['sport'].astype(int)

df['n_pkt_src'] = df['n_pkt_src'].astype(int)

df['n_pkt_dst'] = df['n_pkt_dst'].astype(int)

In [53]:
# Let's again check uniqueness of each feature
for column in df.columns:
    unique_count = df[column].nunique(dropna=False)  
    unique_values = df[column].unique()
    
    print(f"Feature: {column}")
    print(f"Number of Unique Values: {unique_count}")
    print(f"Unique Values: {unique_values}\n")

Feature: mac_s
Number of Unique Values: 9
Unique Values: [0 1 2 3 4 5 6 7 8]

Feature: mac_d
Number of Unique Values: 10
Unique Values: [0 1 2 3 4 5 6 7 8 9]

Feature: ip_s
Number of Unique Values: 9
Unique Values: [ 0  1  2  3  4  5  6 -1  7]

Feature: ip_d
Number of Unique Values: 9
Unique Values: [ 0  1  2  3  4  5  6 -1  7]

Feature: sport
Number of Unique Values: 65537
Unique Values: [56667 56666 56668 ...  1129  1130  1131]

Feature: proto
Number of Unique Values: 5
Unique Values: [0 1 2 3 4]

Feature: flags
Number of Unique Values: 13
Unique Values: [ 0  1  2  3  4  5  6 -1  7  8  9 10 11]

Feature: size
Number of Unique Values: 54
Unique Values: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53]

Feature: modbus_fn
Number of Unique Values: 5
Unique Values: [ 0  1  2  3 -1]

Feature: n_pkt_src
Number of Unique Values: 102
Unique Values: [  0   1   2   3   4   5   6 

In [29]:
# Features that contain missing values and that will be used to construct missing_pattern feature
print([col for col in df.columns if (df[col] == -1).any()])

['ip_s', 'ip_d', 'sport', 'flags', 'modbus_fn', 'n_pkt_src', 'n_pkt_dst', 'modbus_response']


In [30]:
# Function to create a binary string for the missingness pattern
def create_missing_pattern(row, columns):
    binary_string = ''.join(['1' if row[col] == -1 else '0' for col in columns])
    return int(binary_string, 2)

# list of columns with nan values
list_nan_cols = ['ip_s', 'ip_d', 'sport', 'flags', 'modbus_fn', 'n_pkt_src', 'n_pkt_dst', 'modbus_response']

# Apply the function to create the missing_pattern column
df['missing_pattern'] = df.apply(create_missing_pattern, columns=list_nan_cols, axis=1)

In [56]:
# Let's again check uniqueness of each feature
for column in df.columns:
    unique_count = df[column].nunique(dropna=False)  
    unique_values = df[column].unique()
    
    print(f"Feature: {column}")
    print(f"Number of Unique Values: {unique_count}")
    print(f"Unique Values: {unique_values}\n")
    

# Here we notice that the uniqueness of missing_pattern is only 5 which is pretty fascinating :

# Pattern: 0, Binary: 00000000, Missing Columns: (No missing values)

# Pattern: 1, Binary: 00000001, Missing Columns: (1 column missing, modbus_response)

# Pattern: 9, Binary: 00001001, Missing Columns: (2 columns missing, ip_s and modbus_response)

# Pattern: 57, Binary: 00111001, Missing Columns: (5 columns missing, ip_s, flags, modbus_fn, n_pkt_src, modbus_response)

# Pattern: 255, Binary: 11111111, Missing Columns: (All 8 columns missing)


Feature: mac_s
Number of Unique Values: 9
Unique Values: [0 1 2 3 4 5 6 7 8]

Feature: mac_d
Number of Unique Values: 10
Unique Values: [0 1 2 3 4 5 6 7 8 9]

Feature: ip_s
Number of Unique Values: 9
Unique Values: [ 0  1  2  3  4  5  6 -1  7]

Feature: ip_d
Number of Unique Values: 9
Unique Values: [ 0  1  2  3  4  5  6 -1  7]

Feature: sport
Number of Unique Values: 65537
Unique Values: [56667 56666 56668 ...  1129  1130  1131]

Feature: proto
Number of Unique Values: 5
Unique Values: [0 1 2 3 4]

Feature: flags
Number of Unique Values: 13
Unique Values: [ 0  1  2  3  4  5  6 -1  7  8  9 10 11]

Feature: size
Number of Unique Values: 54
Unique Values: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53]

Feature: modbus_fn
Number of Unique Values: 5
Unique Values: [ 0  1  2  3 -1]

Feature: n_pkt_src
Number of Unique Values: 102
Unique Values: [  0   1   2   3   4   5   6 

In [31]:
df.head()

,mac_s,mac_d,ip_s,ip_d,sport,proto,flags,size,modbus_fn,n_pkt_src,...,Valv_15,Valv_16,Valv_17,Valv_18,Valv_19,Valv_20,Valv_21,Valv_22,Label,missing_pattern
0,0,0,0,0,56667,0,0,0,0,0,...,False,False,False,False,False,False,False,False,0,1
1,0,1,0,1,56666,0,0,0,0,1,...,False,False,False,False,False,False,False,False,0,1
2,0,2,0,2,56668,0,0,0,0,2,...,False,False,False,False,False,False,False,False,0,1
3,1,3,1,3,502,0,0,1,1,0,...,False,False,False,False,False,False,False,False,0,0
4,2,3,2,3,502,0,0,1,1,0,...,False,False,False,False,False,False,False,False,0,0


In [32]:
# CHECKPOINT 

# Go see Prepro and Embedding 
df.to_csv('super_checkpoint.csv', index=False)


In [33]:
# We rearranged the order of features in order to give meaning to the new engineered feature 'missing_pattern'.
# This feature was created to capture patterns of NaNs in the dataset.
# By arranging the columns in a specific order, the 'missing_pattern' feature generates a binary value following that order,
# which is then converted to a decimal number, giving each pattern a unique identifier.

new_col_order= ['ip_s', 'ip_d', 'sport', 'flags', 'modbus_fn', 'n_pkt_src', 'n_pkt_dst', 
                'modbus_response', 'missing_pattern', 'mac_s', 'mac_d', 'proto', 'size',
                'Tank_1', 'Tank_2', 'Tank_3', 'Tank_4', 'Tank_5', 'Tank_6', 'Tank_7',
       'Tank_8', 'Pump_1', 'Pump_2', 'Pump_3', 'Pump_4', 'Pump_5', 'Pump_6',
       'Flow_sensor_1', 'Flow_sensor_2', 'Flow_sensor_3', 'Flow_sensor_4',
       'Valv_1', 'Valv_2', 'Valv_3', 'Valv_4', 'Valv_5', 'Valv_6', 'Valv_7',
       'Valv_8', 'Valv_9', 'Valv_10', 'Valv_11', 'Valv_12', 'Valv_13',
       'Valv_14', 'Valv_15', 'Valv_16', 'Valv_17', 'Valv_18', 'Valv_19',
       'Valv_20', 'Valv_21', 'Valv_22', 'Label']

df = df[new_col_order]

In [34]:
print(df.head())

   ip_s  ip_d  sport  flags  modbus_fn  n_pkt_src  n_pkt_dst  modbus_response  \
0     0     0  56667      0          0          0          0               -1   
1     0     1  56666      0          0          1          0               -1   
2     0     2  56668      0          0          2          0               -1   
3     1     3    502      0          1          0          0                0   
4     2     3    502      0          1          0          1                0   

   missing_pattern  mac_s  ...  Valv_14  Valv_15  Valv_16  Valv_17  Valv_18  \
0                1      0  ...    False    False    False    False    False   
1                1      0  ...    False    False    False    False    False   
2                1      0  ...    False    False    False    False    False   
3                0      1  ...    False    False    False    False    False   
4                0      2  ...    False    False    False    False    False   

   Valv_19  Valv_20  Valv_21  Valv_22 

### 2- Physical Data

In [35]:
# Display features of out dataset
print(df.columns)

Index(['ip_s', 'ip_d', 'sport', 'flags', 'modbus_fn', 'n_pkt_src', 'n_pkt_dst',
       'modbus_response', 'missing_pattern', 'mac_s', 'mac_d', 'proto', 'size',
       'Tank_1', 'Tank_2', 'Tank_3', 'Tank_4', 'Tank_5', 'Tank_6', 'Tank_7',
       'Tank_8', 'Pump_1', 'Pump_2', 'Pump_3', 'Pump_4', 'Pump_5', 'Pump_6',
       'Flow_sensor_1', 'Flow_sensor_2', 'Flow_sensor_3', 'Flow_sensor_4',
       'Valv_1', 'Valv_2', 'Valv_3', 'Valv_4', 'Valv_5', 'Valv_6', 'Valv_7',
       'Valv_8', 'Valv_9', 'Valv_10', 'Valv_11', 'Valv_12', 'Valv_13',
       'Valv_14', 'Valv_15', 'Valv_16', 'Valv_17', 'Valv_18', 'Valv_19',
       'Valv_20', 'Valv_21', 'Valv_22', 'Label'],
      dtype='object')


In [36]:
# Display all nan values per feature
print(df.isna().sum())

# We notice no nan values from the plcs data

ip_s               0
ip_d               0
sport              0
flags              0
modbus_fn          0
n_pkt_src          0
n_pkt_dst          0
modbus_response    0
missing_pattern    0
mac_s              0
mac_d              0
proto              0
size               0
Tank_1             0
Tank_2             0
Tank_3             0
Tank_4             0
Tank_5             0
Tank_6             0
Tank_7             0
Tank_8             0
Pump_1             0
Pump_2             0
Pump_3             0
Pump_4             0
Pump_5             0
Pump_6             0
Flow_sensor_1      0
Flow_sensor_2      0
Flow_sensor_3      0
Flow_sensor_4      0
Valv_1             0
Valv_2             0
Valv_3             0
Valv_4             0
Valv_5             0
Valv_6             0
Valv_7             0
Valv_8             0
Valv_9             0
Valv_10            0
Valv_11            0
Valv_12            0
Valv_13            0
Valv_14            0
Valv_15            0
Valv_16            0
Valv_17      

In [61]:
# Clean column names by removing spaces that will later create errors in calls
df.columns = df.columns.str.replace(' ', '')

In [37]:
# Convert boolean features to binary 
boolean_features = df.select_dtypes(include=['bool']).columns.tolist()
continuous_features = ['Tank_1', 'Tank_2', 'Tank_3', 'Tank_4', 'Tank_5', 'Tank_6', 'Tank_7',
                       'Tank_8', 'Flow_sensor_1', 'Flow_sensor_2', 'Flow_sensor_3', 'Flow_sensor_4']
df[boolean_features] = df[boolean_features].astype(int)

In [38]:
print('boolean features :',boolean_features)
print('continuous features :',continuous_features)

boolean features : ['Pump_1', 'Pump_2', 'Pump_3', 'Pump_4', 'Pump_5', 'Pump_6', 'Valv_1', 'Valv_2', 'Valv_3', 'Valv_4', 'Valv_5', 'Valv_6', 'Valv_7', 'Valv_8', 'Valv_9', 'Valv_10', 'Valv_11', 'Valv_12', 'Valv_13', 'Valv_14', 'Valv_15', 'Valv_16', 'Valv_17', 'Valv_18', 'Valv_19', 'Valv_20', 'Valv_21', 'Valv_22']
continuous features : ['Tank_1', 'Tank_2', 'Tank_3', 'Tank_4', 'Tank_5', 'Tank_6', 'Tank_7', 'Tank_8', 'Flow_sensor_1', 'Flow_sensor_2', 'Flow_sensor_3', 'Flow_sensor_4']


In [39]:
# Now all our features are int except for label multiclass
print(df.dtypes)

ip_s               int64
ip_d               int64
sport              int64
flags              int64
modbus_fn          int64
n_pkt_src          int64
n_pkt_dst          int64
modbus_response    int64
missing_pattern    int64
mac_s              int64
mac_d              int64
proto              int64
size               int64
Tank_1             int64
Tank_2             int64
Tank_3             int64
Tank_4             int64
Tank_5             int64
Tank_6             int64
Tank_7             int64
Tank_8             int64
Pump_1             int64
Pump_2             int64
Pump_3             int64
Pump_4             int64
Pump_5             int64
Pump_6             int64
Flow_sensor_1      int64
Flow_sensor_2      int64
Flow_sensor_3      int64
Flow_sensor_4      int64
Valv_1             int64
Valv_2             int64
Valv_3             int64
Valv_4             int64
Valv_5             int64
Valv_6             int64
Valv_7             int64
Valv_8             int64
Valv_9             int64


In [37]:
# Now we extract the continious features to perform normalize them    # DO NOT EXE !!

# List of all features
all_features = df.columns.tolist()

# Continuous features by excluding boolean features
continuous_features = [feature for feature in all_features if feature not in boolean_features]

# Get rid of Time and Label_n and Label
continuous_features = continuous_features[1:-2]

In [40]:
# Initialize the MinMaxScaler to normalize cont data
scaler = MinMaxScaler()

# Normalize the continuous features
df[continuous_features] = scaler.fit_transform(df[continuous_features])

In [41]:
# Let's again check uniqueness of each feature
for column in df.columns:
    unique_count = df[column].nunique(dropna=False)  
    unique_values = df[column].unique()
    
    print(f"Feature: {column}")
    print(f"Number of Unique Values: {unique_count}")
    print(f"Unique Values: {unique_values}\n")

Feature: ip_s
Number of Unique Values: 9
Unique Values: [ 0  1  2  3  4  5  6 -1  7]

Feature: ip_d
Number of Unique Values: 9
Unique Values: [ 0  1  2  3  4  5  6 -1  7]

Feature: sport
Number of Unique Values: 65537
Unique Values: [56667 56666 56668 ...  1129  1130  1131]

Feature: flags
Number of Unique Values: 13
Unique Values: [ 0  1  2  3  4  5  6 -1  7  8  9 10 11]

Feature: modbus_fn
Number of Unique Values: 5
Unique Values: [ 0  1  2  3 -1]

Feature: n_pkt_src
Number of Unique Values: 102
Unique Values: [  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  -1  52
  53  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70
  71  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88
  89  90  91  92  93  94  95  96  97  98  99 100]

Feature: n_pkt_dst
Number of Unique Values: 102
Unique Values: [  

In [42]:
# CHECKPOINT 

# Go see Prepro and Embedding 
df.to_csv('super_checkpoint.csv', index=False)


In [2]:
# load the checkpoint due to crashing kernel
df = pd.read_csv('super_checkpoint.csv')

### 3- Embedding Network Categorical Data

In [3]:
import torch
import torch.nn as nn

In [4]:
# Separate cat features and cont features and labels and drop them
features = df.drop(columns=['Tank_1', 'Tank_2', 'Tank_3', 'Tank_4', 'Tank_5', 'Tank_6',
       'Tank_7', 'Tank_8', 'Pump_1', 'Pump_2', 'Pump_3', 'Pump_4', 'Pump_5',
       'Pump_6', 'Flow_sensor_1', 'Flow_sensor_2', 'Flow_sensor_3',
       'Flow_sensor_4', 'Valv_1', 'Valv_2', 'Valv_3', 'Valv_4', 'Valv_5',
       'Valv_6', 'Valv_7', 'Valv_8', 'Valv_9', 'Valv_10', 'Valv_11', 'Valv_12',
       'Valv_13', 'Valv_14', 'Valv_15', 'Valv_16', 'Valv_17', 'Valv_18',
       'Valv_19', 'Valv_20', 'Valv_21', 'Valv_22','Label'])

In [5]:
features.columns

Index(['ip_s', 'ip_d', 'sport', 'flags', 'modbus_fn', 'n_pkt_src', 'n_pkt_dst',
       'modbus_response', 'missing_pattern', 'mac_s', 'mac_d', 'proto',
       'size'],
      dtype='object')

In [6]:
# Checking if dim has been altered due to error in dimension during embedding
print(np.shape(features))
#print(np.shape(features.values))
#print(np.shape(features_tensor))

(21561878, 13)


In [5]:
# Add 1 to the columns that contain -1 to move away from negative because embedding processes positive only
list_nan_cols = ['ip_s', 'ip_d', 'sport', 'flags', 'modbus_fn', 'n_pkt_src', 'n_pkt_dst', 'modbus_response']
for column in list_nan_cols:
    if (features[column] == -1).any():
        features[column] += 1

In [49]:
# Let's again check uniqueness of each feature for us to gather it data for embedding
for column in features.columns:
    unique_count = features[column].nunique(dropna=False)  
    unique_values = features[column].unique()
    
    print(f"Feature: {column}")
    print(f"Number of Unique Values: {unique_count}")
    print(f"Unique Values: {unique_values}\n")
    
# as we notice  -1 disappeared lettign place to 0

Feature: ip_s
Number of Unique Values: 9
Unique Values: [1 2 3 4 5 6 7 0 8]

Feature: ip_d
Number of Unique Values: 9
Unique Values: [1 2 3 4 5 6 7 0 8]

Feature: sport
Number of Unique Values: 65537
Unique Values: [56668 56667 56669 ...  1130  1131  1132]

Feature: flags
Number of Unique Values: 13
Unique Values: [ 1  2  3  4  5  6  7  0  8  9 10 11 12]

Feature: modbus_fn
Number of Unique Values: 5
Unique Values: [1 2 3 4 0]

Feature: n_pkt_src
Number of Unique Values: 102
Unique Values: [  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52   0  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101]

Feature: n_pkt_dst
Number of Unique Values: 102
Unique Values: [  1   2   3   4   5   6  

In [6]:
# Determine the number of unique values for each feature
unique_counts = features.nunique()

# Define embedding sizes based on uniqueness or what we rather call rule of thumb
embedding_sizes = [(unique, min(50, unique // 2)) for unique in unique_counts]

In [51]:
# make sure that uniqueness list is correct and involves features only
print(unique_counts)

ip_s                   9
ip_d                   9
sport              65537
flags                 13
modbus_fn              5
n_pkt_src            102
n_pkt_dst            102
modbus_response     3302
missing_pattern        5
mac_s                  9
mac_d                 10
proto                  5
size                  54
dtype: int64


In [9]:
# Make sure the embedding mapping is correct
print(embedding_sizes)
print(len(embedding_sizes))

#features['sport'].max()

[(9, 4), (9, 4), (65537, 50), (13, 6), (5, 2), (102, 50), (102, 50), (3302, 50), (5, 2), (9, 4), (10, 5), (5, 2), (54, 27)]
13


In [7]:
# THIS BLOCK TRANSFORMS VALUES TO MATCH MAX TO UNIQUENESS, because we might have uniquenes sof 8000 and max value of 65000 like in sport which wiill
# give problem when embedding
def factorize_sport(series):
    # Identify the values to be factorized (excluding -1)
    mask = series != -1
    
    # Factorize the masked values
    factorized_values, unique_values = pd.factorize(series[mask])
    
    # Create a full series with -1 preserved
    result = pd.Series(-1, index=series.index)
    result[mask] = factorized_values
    return result
features['sport'] = factorize_sport(features['sport'])

In [8]:
#  THIS BLOCK TRANSFORMS VALUES TO MATCH MAX TO UNIQUENESS, same for modbus_resp
def factorize_modbus_response(series):
    # Identify the values to be factorized (excluding -1)
    mask = series != -1
    
    # Factorize the masked values
    factorized_values, unique_values = pd.factorize(series[mask])
    
    # Create a full series with -1 preserved
    result = pd.Series(-1, index=series.index)
    result[mask] = factorized_values
    return result
features['modbus_response'] = factorize_modbus_response(features['modbus_response'])

In [9]:
#  THIS BLOCK TRANSFORMS VALUES TO MATCH MAX TO UNIQUENESS, same for missing pattern, although here i have a question
# here the unique values of this feature exp 255 and 57 are decimal from binaries that represents missing patterns
# accomodating their uniqueness to thier max might result into info loss or more complicated pattern to catch by model
# Thats why in our case and in case of bad results we will set it back to original and augment embedding output foir this feature to 50 instead of 2
# current : [5,2]   change next : [5,50] because of that 255 value
# Factorize the missing_pattern column
missing_pattern_factorized, missing_pattern_unique = pd.factorize(features['missing_pattern'])

# Overwrite column like we did in all previous ones
features['missing_pattern'] = missing_pattern_factorized

In [13]:
#  THIS BLOCK TRANSFORMS VALUES TO MATCH MAX TO UNIQUENESS, same drill
print(features['sport'].max())
print(features['modbus_response'].max())
print(features['missing_pattern'].max())

# as we can see those 3 features gave us problems during embedding due to their max value surpassing, now they re cool, 
# compare them to uniquenss must be lower

65536
3301
4


In [10]:
# Convert features to tensor on order for it to be computed correctly to embeddings
features_tensor = torch.tensor(features.values, dtype=torch.long)

In [11]:
len(features_tensor)

21561878

In [12]:
# Create an embedding layer for each feature and concatenate the embeddings to conserve the same order on the dataset
class EmbeddingNet(nn.Module):
    def __init__(self, embedding_sizes):
        super(EmbeddingNet, self).__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(input_dim, output_dim) for input_dim, output_dim in embedding_sizes])
    
    def forward(self, x):
        embedded_features = []
        for i, embedding in enumerate(self.embeddings):
            # Check the shape and values of x[:, i]
            print(f"Processing feature {i} with shape {x[:, i].shape} and values: {x[:, i]}")
            
            # Ensure that no value in x[:, i] exceeds the embedding's input dimension
            max_index = torch.max(x[:, i]).item()
            min_index = torch.min(x[:, i]).item()
            input_dim = embedding.num_embeddings
            print(f"Feature {i}: min index {min_index}, max index {max_index}, input_dim {input_dim}")
            if max_index >= input_dim or min_index < 0:
                raise ValueError(f"Feature {i} has an index {min_index}-{max_index} out of range for the embedding size {input_dim}")
            
            # Embed the feature
            embedded_feature = embedding(x[:, i])
            print(f"Feature {i} embedded shape: {embedded_feature.shape}")
            embedded_features.append(embedded_feature)
        
        # Concatenate all embedded features
        x = torch.cat(embedded_features, dim=1)
        return x

In [13]:
# Instantiate the model
model = EmbeddingNet(embedding_sizes)

In [14]:
# Forward pass through the model to get embeddings
with torch.no_grad():  # No need to compute gradients because we re not intersted in training
    transformed_features = model(features_tensor)

Processing feature 0 with shape torch.Size([21561878]) and values: tensor([1, 1, 1,  ..., 1, 1, 1])
Feature 0: min index 0, max index 8, input_dim 9
Feature 0 embedded shape: torch.Size([21561878, 4])
Processing feature 1 with shape torch.Size([21561878]) and values: tensor([1, 2, 3,  ..., 5, 5, 5])
Feature 1: min index 0, max index 8, input_dim 9
Feature 1 embedded shape: torch.Size([21561878, 4])
Processing feature 2 with shape torch.Size([21561878]) and values: tensor([    0,     1,     2,  ..., 38298, 38297, 38299])
Feature 2: min index 0, max index 65536, input_dim 65537
Feature 2 embedded shape: torch.Size([21561878, 50])
Processing feature 3 with shape torch.Size([21561878]) and values: tensor([1, 1, 1,  ..., 2, 2, 2])
Feature 3: min index 0, max index 12, input_dim 13
Feature 3 embedded shape: torch.Size([21561878, 6])
Processing feature 4 with shape torch.Size([21561878]) and values: tensor([1, 1, 1,  ..., 0, 0, 0])
Feature 4: min index 0, max index 4, input_dim 5
Feature 4 em

In [15]:
# Convert embeddings to DataFrame
transformed_df = pd.DataFrame(transformed_features.numpy())

In [17]:
# we can see that the embedding was done perfectly no value lost
print(transformed_df.isnull().values.any())

False


In [18]:
transformed_df.head()

,0,1,2,3,4,5,6,7,8,9,...,246,247,248,249,250,251,252,253,254,255
0,-0.100433,0.831530,-0.590548,-1.275605,0.125473,-0.692483,-0.865088,-0.862070,0.440997,0.126347,...,-0.417799,2.261141,-0.539327,0.507017,-0.478598,1.407057,0.093644,-1.812366,0.689168,-0.111608
1,-0.100433,0.831530,-0.590548,-1.275605,-1.427560,-2.214863,2.212478,1.388912,-1.038465,0.176620,...,-0.417799,2.261141,-0.539327,0.507017,-0.478598,1.407057,0.093644,-1.812366,0.689168,-0.111608
2,-0.100433,0.831530,-0.590548,-1.275605,0.149928,-1.317941,-1.305659,0.559081,0.123415,-0.538616,...,-0.417799,2.261141,-0.539327,0.507017,-0.478598,1.407057,0.093644,-1.812366,0.689168,-0.111608
3,0.132813,-1.293596,1.001984,0.162317,0.793673,-0.051979,-0.763262,-0.481465,2.083035,-1.013654,...,-0.448095,1.341901,-2.118302,0.123893,-0.503124,-1.901845,-0.927061,-0.634046,-0.280395,-2.095516
4,1.051991,1.120166,1.191431,1.077091,0.793673,-0.051979,-0.763262,-0.481465,2.083035,-1.013654,...,-0.448095,1.341901,-2.118302,0.123893,-0.503124,-1.901845,-0.927061,-0.634046,-0.280395,-2.095516


In [19]:
# CHECKPOINT of transformed_df to free ram for it then to merge cont features and label from df

# Go see Prepro and Embedding 
transformed_df.to_csv('emb_net_checkpoint.csv', index=False)


In [19]:
# Now this new transformed_df is embedded but misses the phy features and Label that i both removed earlier 
# now we concat the phy features
# create df with phy features only and Label, we drop the features used earlier that are net features
phy = df.drop(columns=features)


In [20]:
print(phy.isnull().values.any())

False


In [21]:
net = transformed_df

In [21]:
# CHECKPOINT of phy df and labels

# Go see Prepro and Embedding 
phy.to_csv('phy_checkpoint.csv', index=False)

In [23]:
# we can see that the embedding is still good no value lost
# this operation is done and repeat to invistigate the sudden nan that appeared on first emb data
print(transformed_df.isnull().values.any())

False


In [24]:
# similarly we check labels and phy data and both are safe til now
print(phy.isnull().values.any())

False


In [ ]:
# Load both emb_net and phy+labels and use numpy to merge them to make it easier for ram to do store
net = pd.read_csv('emb_net_checkpoint.csv')
phy = pd.read_csv('phy_checkpoint.csv')

# it fails to load them together !!!! so we load separately , DONT RUN THIS !!

In [3]:
# Load phy and labels
phy = pd.read_csv('emb_phy/phy_checkpoint.csv')

In [4]:
# checking for nans in it just in case, RAS
print(phy.isnull().values.any())

False


In [22]:
# Transforming phy + labels to numpy 
array_phy = phy.to_numpy()

In [23]:
# Delete the pandas dataset to free memory, keep the numpys data
del phy
# notices big drop in memory usage

In [ ]:
del df

In [ ]:
# Load embedded net 
net = pd.read_csv('emb_phy/emb_net_checkpoint.csv')

In [26]:
# Transforming embedded net to numpy 
array_net = net.to_numpy()

In [27]:
del net, transformed_df

In [ ]:
# Concatenating using numpy np.concatenate

df = np.concatenate([array_net, array_phy], axis=1)

In [ ]:
# concat embedded and phy and labels
transformed_df = pd.concat([transformed_df,phy], axis=1)

In [5]:
a= transformed_df.columns
print(list(a))

NameError: name 'transformed_df' is not defined

In [115]:
transformed_df['Label'].value_counts()

Label
0.0    2687340
1.0    1457507
Name: count, dtype: int64

In [111]:
# This resulted to a df with 296 features
len(transformed_df.columns)

297

In [112]:
# Save the DataFrame to a CSV file in the Kaggle working directory
transformed_df.to_csv('super_embd.csv', index=False)

In [2]:
df = pd.read_csv('super_embd.csv')

In [4]:
df.columns = df.columns.astype(str)
# here we fixed a problem that caused index confusing during training because columlns names was int, now turned it into str

In [6]:
a= df.columns
print(list(a))

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152', '153', '154', '155', '156', '157', '15

In [7]:
df.to_csv('super_embd.csv', index=False)

In [1]:
import pandas as pd
df = pd.read_csv('super_embd.csv')

In [4]:
print(df['Label'].isna().sum())

3197122


In [9]:
len(df['Label'])

7341969

In [10]:
len(df)

7341969

### 4- Checking Time compatibility for merging

#### 1 st catch

In [ ]:
# Convert all entries to datetime, parsing them flexibly
df1['Time'] = pd.to_datetime(df1['Time'], errors='coerce')

# Format all datetime objects to a uniform string format, including microseconds
df1['Formatted_Time'] = df1['Time'].dt.strftime('%Y-%m-%d %H:%M:%S.%f')

# If you want to truncate the microseconds where they are not originally present:
df1['Formatted_Time'] = df1['Formatted_Time'].apply(lambda x: x.rstrip('0').rstrip('.') if '.' in x else x)

print(df1['Formatted_Time'])

In [14]:
df1['Time'] = pd.to_datetime(df1['Time'])  # Convert to datetime if not already
df1['Time_seconds'] = df1['Time'].dt.floor('S')  # Floor to nearest second

# Count unique seconds
unique_seconds_count = df1['Time_seconds'].nunique()

print("There are", unique_seconds_count, "unique seconds in the Time column.")


There are 2420 unique seconds in the Time column.


In [ ]:
# phy1 reset button
df_phy1 = pd.read_csv('Physical_dataset/phy_att_1.csv', encoding='utf-16', delimiter='\t')

In [22]:
len(df_phy1)

2420

In [23]:
print(df1['Time'].iloc[0])
print(df1['Time'].iloc[-1])

2021-04-09 18:23:28.385003
2021-04-09 19:03:47.661291


In [24]:
print(df_phy1['Time'].iloc[0])
print(df_phy1['Time'].iloc[-1])

09/04/2021 18:23:28
09/04/2021 19:03:47


#### 2 nd catch

In [60]:
df = pd.read_csv('/kaggle/input/remainder-net/attack_2.csv')

In [ ]:
# Convert all entries to datetime, parsing them flexibly
df2['Time'] = pd.to_datetime(df2['Time'], errors='coerce')

# Format all datetime objects to a uniform string format, including microseconds
df2['Formatted_Time'] = df2['Time'].dt.strftime('%Y-%m-%d %H:%M:%S.%f')

# If you want to truncate the microseconds where they are not originally present:
df2['Formatted_Time'] = df2['Formatted_Time'].apply(lambda x: x.rstrip('0').rstrip('.') if '.' in x else x)

print(df['Formatted_Time'])

In [26]:
df2['Time'] = pd.to_datetime(df2['Time'])  # Convert to datetime if not already
df2['Time_seconds'] = df2['Time'].dt.floor('S')  # Floor to nearest second

# Count unique seconds
unique_seconds_count = df2['Time_seconds'].nunique()

print("There are", unique_seconds_count, "unique seconds in the Time column.")


There are 2096 unique seconds in the Time column.


In [71]:
# phy2 reset button
df_phy2 = pd.read_csv('Physical_dataset/phy_att_2.csv', encoding='utf-16', delimiter='\t')

In [27]:
len(df_phy2)

# we notice here a  delay of 8, !!!!!! not explainable because there is only a delay of 7 rows which is explainable under because phy_df has 7 seconds before 12 and 2 second after 16 
# compared to df which gives us 7+2 = 9   then once second must be missing inside the net df find it and delete it from phy
# and thats what we will be doing al this time deleting and adjusting rows in phy to suit net before any replication

2104

In [28]:
print(df2['Time'].iloc[0])
print(df2['Time'].iloc[-1])

2021-04-19 15:37:19.989214
2021-04-19 16:12:14.167723


In [29]:
print(df_phy2['Time'].iloc[0])
print(df_phy2['Time'].iloc[-1])

19/04/2021 15:37:12
19/04/2021 16:12:16


In [31]:
df_phy2 = df_phy2.iloc[6:-2]

In [32]:
len(df_phy2) # problem solved DONE

2096

In [33]:
print(df_phy2['Time'].iloc[0])
print(df_phy2['Time'].iloc[-1])

# we notice that the range got settled here like network data

19/04/2021 15:37:19
19/04/2021 16:12:14


In [173]:
# fixing little issue with label name from Lable_n to Label_n

df_phy2 = df_phy2.rename(columns={'Lable_n' : 'Label_n'})

#### 3 rd catch

In [35]:
df = pd.read_csv('/kaggle/input/remainder-net/attack_3.csv')

In [ ]:
# Convert all entries to datetime, parsing them flexibly
df3['Time'] = pd.to_datetime(df3['Time'], errors='coerce')

# Format all datetime objects to a uniform string format, including microseconds
df3['Formatted_Time'] = df3['Time'].dt.strftime('%Y-%m-%d %H:%M:%S.%f')

# If you want to truncate the microseconds where they are not originally present:
df3['Formatted_Time'] = df3['Formatted_Time'].apply(lambda x: x.rstrip('0').rstrip('.') if '.' in x else x)

print(df3['Formatted_Time'])

In [54]:
df3['Time'] = pd.to_datetime(df3['Time'])  # Convert to datetime if not already
df3['Time_seconds'] = df3['Time'].dt.floor('S')  # Floor to nearest second

# Count unique seconds
unique_seconds_count = df3['Time_seconds'].nunique()

print("There are", unique_seconds_count, "unique seconds in the Time column.")

There are 1252 unique seconds in the Time column.


In [40]:
# phy3 reset button
df_phy3 = pd.read_csv('Physical_dataset/phy_att_3.csv', encoding='utf-16', delimiter='\t')

In [55]:
len(df_phy3)

# we notice 2 delay, easy one to fix because there is only a delay of 2 rows which is explainable under because phy_df has one second before 12 and one second after 05 
# compared to df

1254

In [56]:
print(df3['Time'].iloc[0])
print(df3['Time'].iloc[-1])

2021-04-09 19:42:13.484804
2021-04-09 20:03:04.790765


In [57]:
print(df_phy3['Time'].iloc[0])
print(df_phy3['Time'].iloc[-1])

09/04/2021 19:42:12
09/04/2021 20:03:05


In [58]:
df_phy3 = df_phy3.iloc[1:-1]

In [59]:
len(df_phy3) # problem solved DONE

1252

In [60]:
print(df_phy3['Time'].iloc[0])
print(df_phy3['Time'].iloc[-1])

# we notice that the range got settled here like network data

09/04/2021 19:42:13
09/04/2021 20:03:04


#### 4 th catch

In [46]:
df = pd.read_csv('/kaggle/input/remainder-net/attack_4.csv')

In [ ]:
# Convert all entries to datetime, parsing them flexibly
df4['Time'] = pd.to_datetime(df4['Time'], errors='coerce')

# Format all datetime objects to a uniform string format, including microseconds
df4['Formatted_Time'] = df4['Time'].dt.strftime('%Y-%m-%d %H:%M:%S.%f')

# If you want to truncate the microseconds where they are not originally present:
df4['Formatted_Time'] = df4['Formatted_Time'].apply(lambda x: x.rstrip('0').rstrip('.') if '.' in x else x)

print(df4['Formatted_Time'])

In [68]:
df4['Time'] = pd.to_datetime(df4['Time'])  # Convert to datetime if not already
df4['Time_seconds'] = df4['Time'].dt.floor('S')  # Floor to nearest second

# Count unique seconds
unique_seconds_count = df4['Time_seconds'].nunique()

print("There are", unique_seconds_count, "unique seconds in the Time column.")

There are 1711 unique seconds in the Time column.


In [ ]:
# phy4 reset button
df_phy4 = pd.read_csv('Physical_dataset/phy_att_4.csv')

In [70]:
len(df_phy4)

# we notice here a  delay of 6, explainable because there is only a delay of 6 rows which is explainable under because phy_df has 7 seconds before 18 and 1 second before 54 
# compared to df which gives us 7-1 = 6

1717

In [71]:
print(df4['Time'].iloc[0])
print(df4['Time'].iloc[-1])

2022-02-21 14:45:25.454111
2022-02-21 15:13:55.070978


In [72]:
print(df_phy4['Time'].iloc[0])
print(df_phy4['Time'].iloc[-1])

21/02/2022 14:45:18
21/02/2022 15:13:54


In [73]:
df_phy4 = df_phy4.iloc[7:]

In [74]:
len(df_phy4) # problem not solved PROBLEM A WHOLE SET OF SECOND IN NET SHOULD BE DELETED

1710

In [75]:
print(df_phy4['Time'].iloc[0])
print(df_phy4['Time'].iloc[-1])

# we notice that the range didnt get settled here like network data

21/02/2022 14:45:25
21/02/2022 15:13:54


In [81]:
# here we mentionned the extra second we want it to remove of course all the sniffs done within that specific second will be removed too
net_sec_remove = pd.to_datetime('2022-02-21 15:13:55')

In [85]:
# now we remove all rows of that 55 th extra second in network df4
df4 = df4[~(df4['Time'].dt.floor('S') == net_sec_remove)]

In [86]:
print(df4['Time'].iloc[0])
print(df4['Time'].iloc[-1])

2022-02-21 14:45:25.454111
2022-02-21 15:13:54.997799


In [87]:
# as you can see now problem solved
len(df_phy4)

1710

In [88]:
df4['Time'] = pd.to_datetime(df4['Time'])  # Convert to datetime if not already
df4['Time_seconds'] = df4['Time'].dt.floor('S')  # Floor to nearest second

# Count unique seconds
unique_seconds_count = df4['Time_seconds'].nunique()

print("There are", unique_seconds_count, "unique seconds in the Time column.")

There are 1710 unique seconds in the Time column.


/tmp/ipykernel_123618/3868628142.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df4['Time'] = pd.to_datetime(df4['Time'])  # Convert to datetime if not already
/tmp/ipykernel_123618/3868628142.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df4['Time_seconds'] = df4['Time'].dt.floor('S')  # Floor to nearest second


### Merging test


In [93]:
def align_and_merge(net_df, phy_df):
    
    # Truncate time to the nearest second
    phy_df['Time_truncated'] = phy_df['Time'].dt.floor('S')
    net_df['Time_truncated'] = net_df['Time'].dt.floor('S')
    # Count how many entries in net_df for each second
    counts_per_second = net_df.groupby('Time_truncated').size()
    # Map the counts to phy_df based on the truncated time
    phy_df['repetitions'] = phy_df['Time_truncated'].map(counts_per_second)
    # Ensure all repetition numbers are integers
    phy_df['repetitions'] = phy_df['repetitions'].astype(int)
    # Repeat indices according to the number of repetitions needed
    repeated_indices = np.repeat(phy_df.index, phy_df['repetitions'])
    # Create the new DataFrame with repeated rows
    result_phy_df = phy_df.loc[repeated_indices].reset_index(drop=True)
    # Merge the expanded result_phy_df with net_df on 'Time_truncated'
    result_df = pd.merge(result_phy_df, net_df, on='Time_truncated', how='inner')
    return result_df

In [95]:
def align_and_merge(net_df, phy_df):
    # Make sure Time is correctly formatted
    phy_df['Time'] = pd.to_datetime(phy_df['Time'], errors='coerce')
    net_df['Time'] = pd.to_datetime(net_df['Time'], errors='coerce')
    
    # Truncate time to the nearest second
    phy_df['Time_truncated'] = phy_df['Time'].dt.floor('S')
    net_df['Time_truncated'] = net_df['Time'].dt.floor('S')
    
    # Count how many entries in net_df for each second
    counts_per_second = net_df.groupby('Time_truncated').size()
    
    # Create the list of coefficients
    list_coeff = list(counts_per_second)
    
    # Repeat indices according to list_coeff
    repeated_indices = np.repeat(phy_df.index, list_coeff)
    
    # Create the new DataFrame with repeated rows
    result_phy_df = phy_df.loc[repeated_indices].reset_index(drop=True)

    return result_phy_df

In [110]:
def trimming_net_df(net_df , phy_df):
    # get rid of extra rows in net_df that creates inbalance in merging
    rows_2_keep = net_df.index.isin(phy_df.index)
    net_df = net_df[rows_2_keep]
    return net_df

In [145]:
phuper1 = align_and_merge(df1,df_phy1)

In [114]:
nuper1 = trimming_net_df(df1, phuper1)

In [146]:
len(phuper1)

5527403

In [116]:
len(nuper1)

5527403

In [147]:
# We drop Time because we wont use it it will be replaced by net Time and drop last 3 cols generated during replication
phuper1 = phuper1.iloc[:, 1:-3]

In [148]:
phuper1.head()

,Tank_1,Tank_2,Tank_3,Tank_4,Tank_5,Tank_6,Tank_7,Tank_8,Pump_1,Pump_2,...,Valv_15,Valv_16,Valv_17,Valv_18,Valv_19,Valv_20,Valv_21,Valv_22,Label_n,Label
0,0,0,0,0,0,0,0,0,False,False,...,False,False,False,False,False,False,False,False,0,normal
1,0,0,0,0,0,0,0,0,False,False,...,False,False,False,False,False,False,False,False,0,normal
2,0,0,0,0,0,0,0,0,False,False,...,False,False,False,False,False,False,False,False,0,normal
3,0,0,0,0,0,0,0,0,False,False,...,False,False,False,False,False,False,False,False,0,normal
4,0,0,0,0,0,0,0,0,False,False,...,False,False,False,False,False,False,False,False,0,normal


In [128]:
# drop last 4 cols generated during replication
nuper1 = nuper1.iloc[:, :-4]

In [129]:
nuper1.head()

,Time,mac_s,mac_d,ip_s,ip_d,sport,dport,proto,flags,size,modbus_fn,n_pkt_src,n_pkt_dst,modbus_response,label_n,label
0,2021-04-09 18:23:28.385003,74:46:a0:bd:a7:1b,0a:fe:ec:47:74:fb,84.3.251.20,84.3.251.102,56667.0,502.0,Modbus,11000.0,66,Read Coils Request,0.0,0.0,NaN,0,normal
1,2021-04-09 18:23:28.385005,74:46:a0:bd:a7:1b,e6:3f:ac:c9:a8:8c,84.3.251.20,84.3.251.101,56666.0,502.0,Modbus,11000.0,66,Read Coils Request,1.0,0.0,NaN,0,normal
2,2021-04-09 18:23:28.385006,74:46:a0:bd:a7:1b,fa:00:bc:90:d7:fa,84.3.251.20,84.3.251.103,56668.0,502.0,Modbus,11000.0,66,Read Coils Request,2.0,0.0,NaN,0,normal
3,2021-04-09 18:23:28.385484,0a:fe:ec:47:74:fb,74:46:a0:bd:a7:1b,84.3.251.102,84.3.251.20,502.0,56667.0,Modbus,11000.0,64,Read Coils Response,0.0,0.0,[0],0,normal
4,2021-04-09 18:23:28.385486,fa:00:bc:90:d7:fa,74:46:a0:bd:a7:1b,84.3.251.103,84.3.251.20,502.0,56668.0,Modbus,11000.0,64,Read Coils Response,0.0,1.0,[0],0,normal


In [149]:
# No that everything is setup and ready we will merge 
super1 = pd.merge(nuper1, phuper1, left_index=True, right_index=True, how='inner')

In [154]:
# Clean column names by removing spaces that will later create errors in calls
super1.columns = super1.columns.str.replace(' ', '')

In [156]:
super1 = super1.drop(['label','Label'],axis = 1)

In [157]:
super1.columns

Index(['Time', 'mac_s', 'mac_d', 'ip_s', 'ip_d', 'sport', 'dport', 'proto',
       'flags', 'size', 'modbus_fn', 'n_pkt_src', 'n_pkt_dst',
       'modbus_response', 'label_n', 'Tank_1', 'Tank_2', 'Tank_3', 'Tank_4',
       'Tank_5', 'Tank_6', 'Tank_7', 'Tank_8', 'Pump_1', 'Pump_2', 'Pump_3',
       'Pump_4', 'Pump_5', 'Pump_6', 'Flow_sensor_1', 'Flow_sensor_2',
       'Flow_sensor_3', 'Flow_sensor_4', 'Valv_1', 'Valv_2', 'Valv_3',
       'Valv_4', 'Valv_5', 'Valv_6', 'Valv_7', 'Valv_8', 'Valv_9', 'Valv_10',
       'Valv_11', 'Valv_12', 'Valv_13', 'Valv_14', 'Valv_15', 'Valv_16',
       'Valv_17', 'Valv_18', 'Valv_19', 'Valv_20', 'Valv_21', 'Valv_22',
       'Label_n'],
      dtype='object')

In [164]:
# Here we compute the number of times the merge didnt agree on a labelisation 0/1
count= (super1['label_n'] != super1['Label_n']).sum()

In [163]:
err_perc = (count * 100) / len(super1)
err_perc

0.26728646346213586

Here we notice the error or discrepancy which is only 0.26 % during merging in this first super set

Now that we understood concept we process and merge rest directly


In [174]:
phuper2 = align_and_merge(df2,df_phy2)
nuper2 = trimming_net_df(df2, phuper2)


phuper2 = phuper2.iloc[:, 1:-1]

nuper2 = nuper2.iloc[:, :-3]

super2 = pd.merge(nuper2, phuper2, left_index=True, right_index=True, how='inner')

super2.columns = super2.columns.str.replace(' ', '')

super2 = super2.drop(['label','Label'],axis = 1)

count2= (super2['label_n'] != super2['Label_n']).sum()

err_perc2 = (count2 * 100) / len(super2)
err_perc2

0.5449401158957148

In [175]:
phuper3 = align_and_merge(df3,df_phy3)
nuper3 = trimming_net_df(df3, phuper3)


phuper3 = phuper3.iloc[:, 1:-1]

nuper3 = nuper3.iloc[:, :-3]

super3 = pd.merge(nuper3, phuper3, left_index=True, right_index=True, how='inner')

super3.columns = super3.columns.str.replace(' ', '')

super3 = super3.drop(['label','Label'],axis = 1)

count3= (super3['label_n'] != super3['Label_n']).sum()

err_perc3 = (count3 * 100) / len(super3)
err_perc3

0.274437359915668

In [184]:
phuper4 = align_and_merge(df4,df_phy4)
nuper4 = trimming_net_df(df4, phuper4)


phuper4 = phuper4.iloc[:, 1:-1]

nuper4 = nuper4.iloc[:, :-3]

super4 = pd.merge(nuper4, phuper4, left_index=True, right_index=True, how='inner')

super4.columns = super4.columns.str.replace(' ', '')

super4 = super4.drop(['label','Label'],axis = 1)

count4 = (super4['label_n'] != super4['Label_n']).sum()

err_perc4 = (count4 * 100) / len(super4)
err_perc4

5.032975346670749

As you can witness those are the error percentages by acquisition :
- 1 : 0.26728646346213586
- 2 : 0.5449401158957148
- 3 : 0.274437359915668
- 4 : 5.032975346670749

Now we move to merging labels with or operator to consolidate attack detection, because as mentionned in original paper some attacks wont affect network process and viseversa

In [187]:
super1['Label'] = (( super1['label_n'] == 1) | ( super1['Label_n'] == 1)).astype(int)

In [189]:
super2['Label'] = (( super2['label_n'] == 1) | ( super2['Label_n'] == 1)).astype(int)

In [190]:
super3['Label'] = (( super3['label_n'] == 1) | ( super3['Label_n'] == 1)).astype(int)

In [191]:
super4['Label'] = (( super4['label_n'] == 1) | ( super4['Label_n'] == 1)).astype(int)

In [192]:
super1 = super1.drop(['label_n','Label_n'], axis = 1)
super2 = super2.drop(['label_n','Label_n'], axis = 1)
super3 = super3.drop(['label_n','Label_n'], axis = 1)
super4 = super4.drop(['label_n','Label_n'], axis = 1)

now we save them merged 4 supers

In [197]:
super1.to_csv('new_supers/super1.csv', index=False)
super2.to_csv('new_supers/super2.csv', index=False)
super3.to_csv('new_supers/super3.csv', index=False)
super4.to_csv('new_supers/super4.csv', index=False)

DONE SUCCESSFULLY

Next merge the 4 sups (4-2-1-3) in this order because it is order of acquisition, and proceed to embedding

In [3]:
# Concat the 4 supers following the acquisition time order

sup1 = pd.read_csv('new_supers/super1.csv')
sup2 = pd.read_csv('new_supers/super2.csv')
sup3 = pd.read_csv('new_supers/super3.csv')
sup4 = pd.read_csv('new_supers/super4.csv')

In [4]:
# we concat following the time order
df = pd.concat([sup1, sup3, sup2, sup4], ignore_index=True)

In [5]:
print(df['Label'].isna().sum())

0


In [9]:
# no use for Time for us anymore because all packets are in order of chronology
df = df.drop(['Time'], axis = 1)

In [6]:
len(df)

22071868

In [7]:
# Here we drop duplicates knowing that we have large ammount of data per row witth merging of phy data so we get rid of redundancy
df = df.drop_duplicates()

# since ics is a  cyclical process if i drop duplicates before dropping time this will reduce my dataset distingutively

In [8]:
len(df)
# here we notice a drastic drop in packets from 22 Millions to 4 Millions when dropping ddupli after dropping time thats why we reverse
# and end up with 21 M data points

21561878

In [21]:
# Iterate over each feature to print the number of unique values and the unique values themselves
for column in df.columns:
    unique_count = df[column].nunique(dropna=False)  # Set dropna=False to include NaN in the count if present
    unique_values = df[column].unique()
    
    print(f"Feature: {column}")
    print(f"Number of Unique Values: {unique_count}")
    print(f"Unique Values: {unique_values}\n")

Feature: Time
Number of Unique Values: 20546717
Unique Values: ['2021-04-09 18:23:28.385003' '2021-04-09 18:23:28.385005'
 '2021-04-09 18:23:28.385006' ... '2022-02-21 15:13:54.984002'
 '2022-02-21 15:13:54.987165' '2022-02-21 15:13:54.990367']

Feature: mac_s
Number of Unique Values: 9
Unique Values: ['74:46:a0:bd:a7:1b' '0a:fe:ec:47:74:fb' 'fa:00:bc:90:d7:fa'
 'e6:3f:ac:c9:a8:8c' '00:80:f4:03:fb:12' 'fe:bb:16:7b:c3:27'
 '4a:35:83:e0:3d:a4' '00:0c:29:47:8c:22' '00:0c:29:47:8c:0e']

Feature: mac_d
Number of Unique Values: 10
Unique Values: ['0a:fe:ec:47:74:fb' 'e6:3f:ac:c9:a8:8c' 'fa:00:bc:90:d7:fa'
 '74:46:a0:bd:a7:1b' '00:80:f4:03:fb:12' 'fe:bb:16:7b:c3:27'
 '4a:35:83:e0:3d:a4' 'ff:ff:ff:ff:ff:ff' '00:0c:29:47:8c:22'
 '00:0c:29:47:8c:0e']

Feature: ip_s
Number of Unique Values: 9
Unique Values: ['84.3.251.20' '84.3.251.102' '84.3.251.103' '84.3.251.101' '84.3.251.18'
 '84.3.251.105' '84.3.251.104' nan '84.3.251.110']

Feature: ip_d
Number of Unique Values: 9
Unique Values: ['84.3.251

In [10]:
# Go see Prepro and Embedding 
df.to_csv('super.csv', index=False)

# II / Model

### 1- First Test Vanilla NN

In [2]:
df = transformed_df

NameError: name 'transformed_df' is not defined

In [167]:
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [168]:
# Function to split
def preprocess_data(df):
    # Separate features and labels
    X = df.drop(columns=['label']).values
    y = df['label'].values
    
    # Split the data into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    return X_train, X_test, y_train, y_test


In [169]:
# Create tensors to feed model
def create_tensors(X_train, X_test, y_train, y_test):
    # Convert to PyTorch tensors
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.long)
    y_test_tensor = torch.tensor(y_test, dtype=torch.long)
    
    return X_train_tensor, X_test_tensor, y_train_tensor, y_test_tensor

In [170]:
# Basic vanilla Neural Network
class DeeperNN(nn.Module):
    def __init__(self, input_dim):
        super(DeeperNN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.fc2 = nn.Linear(512, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.fc3 = nn.Linear(256, 128)
        self.bn3 = nn.BatchNorm1d(128)
        self.fc4 = nn.Linear(128, 64)
        self.bn4 = nn.BatchNorm1d(64)
        self.fc5 = nn.Linear(64, 2)  # for binary classification
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x):
        x = torch.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = torch.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)
        x = torch.relu(self.bn3(self.fc3(x)))
        x = self.dropout(x)
        x = torch.relu(self.bn4(self.fc4(x)))
        x = self.fc5(x)
        return x

In [171]:
# Classical training function using Cross entropy loss and adam
def train_model(model, X_train_tensor, y_train_tensor, num_epochs=20, learning_rate=0.001):
    # Define loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # Training loop
    for epoch in range(num_epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train_tensor)
        loss = criterion(outputs, y_train_tensor)
        loss.backward()
        optimizer.step()
        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {loss.item()}')

In [172]:
# Eval function
def evaluate_model(model, X_test_tensor, y_test_tensor):
    # Evaluate the model
    model.eval()
    with torch.no_grad():
        outputs = model(X_test_tensor)
        _, predicted = torch.max(outputs, 1)
        accuracy = (predicted == y_test_tensor).sum().item() / y_test_tensor.size(0)
    return accuracy

In [1]:
# Preprocess data
X_train, X_test, y_train, y_test = preprocess_data(df)

# Create tensors
X_train_tensor, X_test_tensor, y_train_tensor, y_test_tensor = create_tensors(X_train, X_test, y_train, y_test)

# Define and train the model
input_dim = X_train_tensor.shape[1]
model = DeeperNN(input_dim)
train_model(model, X_train_tensor, y_train_tensor)

# Evaluate the model
accuracy = evaluate_model(model, X_test_tensor, y_test_tensor)
print(f'Accuracy on test set: {accuracy * 100:.2f}%')

NameError: name 'preprocess_data' is not defined